# Modded-NanoGPT

This notebook trains a NanoGPT model to use 8 NVIDIA H100 GPUs to attains 3.28 cross-entropy loss on the FineWeb validation set.

The target (3.28 validation loss on FineWeb) follows Andrej Karpathy's GPT-2 replication in llm.c, which attains that loss after running for 45 minutes.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

libraries = [
    "torch",
    "numpy",
]

for dist in libraries:
    try:
        print(f"{dist}: {version(dist)}")
    except PackageNotFoundError:
        print(f"{dist}: <not installed>")


In [10]:
!uname -ar

Linux 209-20-159-132 6.11.0-29-generic #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2 x86_64 x86_64 x86_64 GNU/Linux


In [11]:
import multiprocessing

multiprocessing.cpu_count()

208

In [7]:
# print all the version for the following libraries: pytorc, cuda
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        device = torch.cuda.get_device_properties(i)
        
        print("---"*80)
        print()
        print(f"CUDA {i} Device Properties:")
        print(f"Name: {device.name}")
        print(f"Compute Capability: {device.major}.{device.minor}")
        print(f"Total Memory: {device.total_memory} MB")
        print(f"Multiprocessors: {device.multi_processor_count}")
        print(f"UUID: {device.uuid}")
        print(f"PCI - Bus {device.pci_bus_id}, Device {device.pci_device_id}, Domain {device.pci_domain_id}")
        print(f"L2 Cache Size: {device.L2_cache_size} MB")
else:
    print("WARNING NO VALID CUDA SETUP FOUND")

PyTorch version: 2.8.0+cu128
CUDA version: 12.8
CUDA available: True
CUDA device count: 8
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

CUDA 0 Device Properties:
Name: NVIDIA H100 80GB HBM3
Compute Capability: 9.0
Total Memory: 85028896768 MB
Multiprocessors: 132
UUID: 6152699b-8afe-db77-0e1e-d46e820ea347
PCI - Bus 97, Device 0, Domain 0
L2 Cache Size: 52428800 MB
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

CUDA 1 Device Properties:
Name: NVIDIA H100 80GB HBM3
Compute Capability: 9.0
Total Memory: 85028896768 MB
Multiprocessors: 132
UUID: f0a18acd-37d9-4354-7a03-124f50573722
PCI - Bus 98, Device 0, 

In [8]:
# Setup fast transfer libraries
%env HF_TRANSFER=1
%env HF_HUB_ENABLE_HF_TRANSFER=1

env: HF_TRANSFER=1
env: HF_HUB_ENABLE_HF_TRANSFER=1


In [9]:
import os

DATASET_PATH = os.path.join(os.getcwd(), "data")
os.makedirs(DATASET_PATH, exist_ok=True)
os.environ["DATASET_PATH"] = DATASET_PATH

In [5]:
# download fine web
import os

from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor, as_completed

def get(fname):
    local_dir = os.path.join(os.environ['DATASET_PATH'], 'fineweb10B')
    if not os.path.exists(os.path.join(local_dir, fname)):
        hf_hub_download(
            repo_id="kjj0/fineweb10B-gpt2",
            filename=fname,
            repo_type="dataset",
            local_dir=local_dir
        )

num_chunks = 8  # full fineweb10B use 103. Each chunk is 100M tokens

files = ["fineweb_val_%06d.bin" % 0] + [
    "fineweb_train_%06d.bin" % i for i in range(1, num_chunks + 1)
]

with ThreadPoolExecutor(max_workers=8) as executor:  # adjust workers
    futures = {executor.submit(get, f): f for f in files}
    for future in as_completed(futures):
        fname = futures[future]
        try:
            future.result()
            print(f"Downloaded {fname}")
        except Exception as e:
            print(f"Failed {fname}: {e}")

fineweb_val_000000.bin:   0%|          | 0.00/200M [00:00<?, ?B/s]

fineweb_train_000007.bin:   0%|          | 0.00/200M [00:00<?, ?B/s]

fineweb_train_000004.bin:   0%|          | 0.00/200M [00:00<?, ?B/s]

fineweb_train_000005.bin:   0%|          | 0.00/200M [00:00<?, ?B/s]

fineweb_train_000001.bin:   0%|          | 0.00/200M [00:00<?, ?B/s]

...

fineweb_train_000006.bin:   0%|          | 0.00/200M [00:00<?, ?B/s]

fineweb_train_000002.bin:   0%|          | 0.00/200M [00:00<?, ?B/s]

Downloaded fineweb_train_000001.bin
Downloaded fineweb_train_000005.bin


fineweb_train_000008.bin:   0%|          | 0.00/200M [00:00<?, ?B/s]

Downloaded fineweb_train_000002.bin
Downloaded fineweb_train_000006.bin
Downloaded fineweb_train_000004.bin
Downloaded fineweb_train_000003.bin
Downloaded fineweb_val_000000.bin
Downloaded fineweb_train_000007.bin
Downloaded fineweb_train_000008.bin
